# NB08: Direct metagenomic functional profiles (H9)

**[EXPLORATORY / POST-HOC]**

Tests **H9**: Direct metagenomic functional profiles (SPIRE MAGs + eggNOG annotations)
predict metal concentrations better than 16S-inferred genus-weighted features from NB02.

**Motivation**: NB02's genus-weighted features (GW) use 16S OTU relative abundances linked
to a pangenome-based trait table. This 16S-to-function bridge introduces noise from:
1. 16S → genus mapping error
2. Genus → pangenome representativeness
3. Aggregation over pan-genome rather than actual MAGs in sample

Direct shotgun metagenomics-derived functional profiles (per-Mb gene density from SPIRE MAGs)
could resolve all three sources of noise if the SPIRE MAG coverage is sufficient.

**Design**:
- Source: SPIRE MAGs with eggNOG functional annotations (`arkinlab.spire.eggnog_annotations_spire`)
- Per-Mb gene density per functional category per sample
- Presence-weighted community profile: `sum(MAG_depth × category_density) / total_depth`
- Model suite: B0, CLR-only (from NB02 feature_matrix), GW-only (NB02), MAG-derived functional profile
- Compare: MAG-derived vs 16S-inferred GW features on spatial block CV

**H9 success criterion**: MAG-derived functional model RMSE < GW-only model RMSE for ≥2 of 4 metals.

**Note**: SPIRE coverage may be limited (only samples with shotgun data). The analysis
is constrained to the intersection of SPIRE-covered samples and HMP feature_matrix.

**Outputs**:
- `data/metagenomic_prediction_results.csv`
- `figures/mag_vs_16s_rmse.png`

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIG_DIR  = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())

from modelling import TARGETS, rmse, _drop_nan_rows, build_xgboost

# Spark connection (for SPIRE MAG data)
try:
    from pyspark.sql import SparkSession
    spark = get_spark_session()
    SPARK_AVAILABLE = True
    print('Spark connected (remote mode).')
except Exception:
    try:
        from pyspark.sql import SparkSession
        spark = get_spark_session()
        SPARK_AVAILABLE = True
        print('Spark connected (local mode).')
    except Exception as e:
        SPARK_AVAILABLE = False
        print(f'Spark not available: {e}')

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
blocks = pd.read_csv(DATA_DIR / 'spatial_blocks.csv', index_col=0)['block']
print(f'Feature matrix: {feature_matrix.shape}')

Spark connected (remote mode).
Feature matrix: (42037, 354)


## 1. Load SPIRE MAG functional annotations via Spark

The SPIRE eggNOG table has one row per MAG × eggNOG annotation. We compute:
- Per-MAG gene density in metal-associated categories (metal_clusters, homeostasis, defense, metabolism)
- Per-sample abundance-weighted mean gene density across all MAGs

In [2]:
if not SPARK_AVAILABLE:
    print('WARNING: Spark not available. NB08 cannot load SPIRE data.')
    print('This notebook requires JupyterHub with Spark Connect to proceed.')
    print('Results will be INCOMPLETE until run on the cluster.')
    mag_features = None
else:
    # SPIRE eggNOG annotations: check table structure
    try:
        spire_eg = spark.table('arkinlab.spire.eggnog_annotations_spire')
        print(f'SPIRE eggNOG: {spire_eg.count():,} rows')
        print('Columns:', spire_eg.columns[:20])
        spire_eg.printSchema()
    except Exception as e:
        print(f'Could not access SPIRE table: {e}')
        mag_features = None
        SPARK_AVAILABLE = False

SPIRE eggNOG: 15,050,686 rows
Columns: ['query', 'seed_ortholog', 'evalue', 'score', 'eggNOG_OGs', 'max_annot_lvl', 'COG_category', 'Description', 'Preferred_name', 'GOs', 'EC', 'KEGG_ko', 'KEGG_Pathway', 'KEGG_Module', 'KEGG_Reaction', 'KEGG_rclass', 'BRITE', 'KEGG_TC', 'CAZy', 'BiGG_Reaction']
root
 |-- query: string (nullable = true)
 |-- seed_ortholog: string (nullable = true)
 |-- evalue: string (nullable = true)
 |-- score: string (nullable = true)
 |-- eggNOG_OGs: string (nullable = true)
 |-- max_annot_lvl: string (nullable = true)
 |-- COG_category: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Preferred_name: string (nullable = true)
 |-- GOs: string (nullable = true)
 |-- EC: string (nullable = true)
 |-- KEGG_ko: string (nullable = true)
 |-- KEGG_Pathway: string (nullable = true)
 |-- KEGG_Module: string (nullable = true)
 |-- KEGG_Reaction: string (nullable = true)
 |-- KEGG_rclass: string (nullable = true)
 |-- BRITE: string (nullable = true)
 

In [3]:
if not SPARK_AVAILABLE or 'spire_eg' not in dir():
    print('Skipping SPIRE processing — Spark unavailable.')
    mag_features = None
else:
    from pyspark.sql import functions as F

    # ── Discover available SPIRE tables ───────────────────────────────────────
    try:
        tables = spark.sql("SHOW TABLES IN arkinlab.spire").toPandas()
        print('Available SPIRE tables:', tables['tableName'].tolist())
    except Exception as e:
        print(f'SHOW TABLES failed: {e}')
        tables = pd.DataFrame()

    # ── Inspect mag_id format ─────────────────────────────────────────────────
    sample_rows = spire_eg.select('query', 'mag_id').limit(3).toPandas()
    print('\nSample mag_id format:')
    print(sample_rows.to_string(index=False))

    # ── Conclusion: data infrastructure limitation ────────────────────────────
    #
    # arkinlab.spire contains ONLY 'eggnog_annotations_spire'.
    # mag_id format is opaque ('spire_mag_XXXXXXXX') — no sample_id embedded.
    # There is no sample→MAG coverage/mapping table in this namespace.
    #
    # Per-sample abundance-weighted functional profiles require:
    #   1. Sample-to-MAG assignment (which MAGs are present in which sample)
    #   2. Per-MAG coverage depth in each sample
    #   3. Per-MAG genome size (for per-Mb normalisation)
    #
    # None of these are available in arkinlab.spire.
    # H9 is UNTESTABLE with the current SPIRE data infrastructure.
    #
    print('\n' + '='*60)
    print('H9 UNTESTABLE: arkinlab.spire lacks sample→MAG coverage table.')
    print('Available tables:', tables['tableName'].tolist() if len(tables) > 0 else ['none'])
    print('mag_id format is opaque — no sample_id embedded.')
    print('Required: sample_mag_coverage table with sample_id, mag_id, depth.')
    print('='*60)
    mag_features = None


Available SPIRE tables: ['eggnog_annotations_spire']



Sample mag_id format:
      query             mag_id
k141_2777_1 spire_mag_01820161
k141_2777_2 spire_mag_01820161
k141_2777_3 spire_mag_01820161

H9 UNTESTABLE: arkinlab.spire lacks sample→MAG coverage table.
Available tables: ['eggnog_annotations_spire']
mag_id format is opaque — no sample_id embedded.
Required: sample_mag_coverage table with sample_id, mag_id, depth.


## 2. Spatial block CV with MAG-derived features

In [4]:
if mag_features is None:
    print('MAG features not available. Skipping spatial CV.')
    print('Re-run after fixing SPIRE table access above.')
    mag_cv = pd.DataFrame()
else:
    # Join MAG features to feature_matrix
    fm_mag = feature_matrix.join(mag_features, how='inner')
    print(f'Intersection (feature_matrix ∩ SPIRE): {len(fm_mag):,} samples')

    MAG_COLS = [c for c in fm_mag.columns if c.startswith('mag_')]
    CLR_COLS = [c for c in fm_mag.columns if c.startswith('clr_')]
    GW_COLS  = [c for c in fm_mag.columns if c.startswith('gw_')]

    MODELS = {
        'B0':  None,
        'GW':  CLR_COLS + GW_COLS,  # NB02 GW features (16S-inferred)
        'MAG': MAG_COLS,             # Direct MAG-derived functional profile
        'MAG+CLR': MAG_COLS + CLR_COLS,
    }

    blocks_mag = blocks.reindex(fm_mag.index)
    mag_results = []

    for target in TARGETS:
        if target not in fm_mag.columns:
            continue
        y = fm_mag[target]
        print(f'\n=== {target} ===')

        for model_name, feature_cols in MODELS.items():
            if model_name == 'B0':
                for test_block in blocks_mag.dropna().unique():
                    train_mask = (blocks_mag != test_block) & blocks_mag.notna()
                    test_mask  = blocks_mag == test_block
                    y_train = y[train_mask].dropna()
                    y_test  = y[test_mask].dropna()
                    if len(y_test) < 5: continue
                    preds = np.full(len(y_test), float(y_train.mean()))
                    mag_results.append({'model': 'B0', 'target': target, 'block': test_block,
                                       'n_train': len(y_train), 'n_test': len(y_test),
                                       'rmse': rmse(y_test.values, preds)})
                continue

            fcols = [c for c in feature_cols if c in fm_mag.columns]
            X = fm_mag[fcols]
            for test_block in blocks_mag.dropna().unique():
                test_mask  = blocks_mag == test_block
                train_mask = ~test_mask & blocks_mag.notna()
                X_tr, y_tr = X[train_mask], y[train_mask]
                X_te, y_te = X[test_mask], y[test_mask]
                valid_tr = ~y_tr.isna() & ~X_tr.isna().any(axis=1)
                valid_te = ~y_te.isna() & ~X_te.isna().any(axis=1)
                X_tr, y_tr = X_tr[valid_tr], y_tr[valid_tr]
                X_te, y_te = X_te[valid_te], y_te[valid_te]
                if len(X_tr) < 20 or len(X_te) < 5: continue
                m = build_xgboost()
                m.fit(X_tr, y_tr)
                preds = m.predict(X_te)
                mag_results.append({'model': model_name, 'target': target, 'block': test_block,
                                   'n_train': len(X_tr), 'n_test': len(X_te),
                                   'rmse': rmse(y_te.values, preds)})
            print(f'  {model_name}: done')

    mag_cv = pd.DataFrame(mag_results)
    print('\nMAG-derived CV RMSE (mean across blocks):')
    print(mag_cv.groupby(['model','target'])['rmse'].mean().unstack('target').round(4))

MAG features not available. Skipping spatial CV.
Re-run after fixing SPIRE table access above.


## 3. H9 evaluation and comparison plot

In [5]:
if len(mag_cv) == 0:
    print('='*60)
    print('H9 OUTCOME: UNTESTABLE')
    print()
    print('Reason: arkinlab.spire contains only gene-level eggNOG annotations')
    print('(eggnog_annotations_spire). The mag_id field is an opaque numeric ID')
    print('("spire_mag_XXXXXXXX") with no sample information embedded.')
    print()
    print('To compute per-sample abundance-weighted MAG functional profiles, the')
    print('following data are required but absent from arkinlab.spire:')
    print('  1. sample→MAG assignment table (which MAGs are in which sample)')
    print('  2. Per-MAG coverage depth per sample')
    print('  3. Per-MAG genome size for per-Mb normalisation')
    print()
    print('H9 reference values (GW-only RMSE from NB02 spatial block CV):')
    print('  Cu: 1.143  Zn: 0.703  Pb: 0.970  Ni: 1.842')
    print()
    print('Next step: request sample_mag_coverage table from SPIRE data team,')
    print('or access shotgun coverage via an alternative route (e.g., CheckM2')
    print('outputs with coverage from CoverM).')
    print('='*60)
else:
    mean_rmse = mag_cv.groupby(['model', 'target'])['rmse'].mean()

    h9_records = []
    for target in TARGETS:
        gw_rmse  = mean_rmse.get(('GW',  target), float('nan'))
        mag_rmse = mean_rmse.get(('MAG', target), float('nan'))
        b0_rmse  = mean_rmse.get(('B0',  target), float('nan'))
        h9_records.append({'target': target, 'B0': b0_rmse, 'GW': gw_rmse,
                           'MAG': mag_rmse, 'mag_beats_gw': mag_rmse < gw_rmse})

    h9_df = pd.DataFrame(h9_records)
    print('H9 comparison (MAG-derived vs GW features):')
    print(h9_df.to_string(index=False))

    n_h9 = int(h9_df['mag_beats_gw'].sum())
    print(f'\nH9 OUTCOME: {"SUPPORTED" if n_h9 >= 2 else "NOT SUPPORTED"} ({n_h9}/4 metals MAG beats GW)')
    print('[NOTE: Exploratory/post-hoc. Positive result warrants pre-registration.]')

    mag_cv.to_csv(DATA_DIR / 'metagenomic_prediction_results.csv', index=False)
    print(f'Saved metagenomic_prediction_results.csv')

    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    axes = axes.flatten()
    for i, target in enumerate(TARGETS):
        ax = axes[i]
        models = ['B0', 'GW', 'MAG', 'MAG+CLR']
        rmse_vals = [mean_rmse.get((m, target), float('nan')) for m in models]
        ax.bar(models, rmse_vals, color=['gray', 'steelblue', 'darkorange', 'green'], alpha=0.8)
        ax.set_title(target.replace('log_', '').replace('_ppm', ''))
        ax.set_ylabel('RMSE')
    plt.suptitle('H9: Direct MAG profiles vs 16S-inferred GW features')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'mag_vs_16s_rmse.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved mag_vs_16s_rmse.png')


H9 OUTCOME: UNTESTABLE

Reason: arkinlab.spire contains only gene-level eggNOG annotations
(eggnog_annotations_spire). The mag_id field is an opaque numeric ID
("spire_mag_XXXXXXXX") with no sample information embedded.

To compute per-sample abundance-weighted MAG functional profiles, the
following data are required but absent from arkinlab.spire:
  1. sample→MAG assignment table (which MAGs are in which sample)
  2. Per-MAG coverage depth per sample
  3. Per-MAG genome size for per-Mb normalisation

H9 reference values (GW-only RMSE from NB02 spatial block CV):
  Cu: 1.143  Zn: 0.703  Pb: 0.970  Ni: 1.842

Next step: request sample_mag_coverage table from SPIRE data team,
or access shotgun coverage via an alternative route (e.g., CheckM2
outputs with coverage from CoverM).
